In [2]:
"""
Climate-Induced Migration Pressure Modeling — India (District-Level)
Step 4: Synthetic Target Variable — migration_pressure_score
"""
 
import sys
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
 
import sys
sys.path.append("C:/Users/asus/OneDrive/Documents/climate-migration-prediction/src")
from step02_preprocessing import (       # noqa: E402  (import after sys.path fix)
    preprocess_census,
    preprocess_rainfall,
    preprocess_agriculture,
    preprocess_mpi,
    CENSUS_PATH, RAINFALL_PATH, MPI_PATH, AGRI_PATH,
)

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# Configuration
# ─────────────────────────────────────────────────────────────────────────────
 
JOIN_KEYS = ["state_key", "district_key"]
 
# Component definitions: (feature_name, higher = more stress)
# worker_ratio and marginal_worker_rate are negated because higher marginal
# worker share = more precarious livelihoods = higher migration pressure.
# worker_ratio is inverted: low formal employment → high pressure.
COMPONENTS = {
    "climate": {
        "features": ["rainfall_cv", "drought_freq", "flood_freq"],
        "weight":   0.35,
    },
    "agriculture": {
        "features": ["yield_cv", "production_std"],
        "weight":   0.25,
    },
    "poverty": {
        "features": ["mpi", "headcount_ratio"],
        "weight":   0.25,
    },
    "demographics": {
        # marginal_worker_rate: higher → more stress (keep as-is)
        # worker_ratio: higher formal employment → LESS stress → invert
        "features": ["marginal_worker_rate", "worker_ratio"],
        "invert":   [False, True],
        "weight":   0.15,
    },
}
 
ALL_FEATURES = [f for comp in COMPONENTS.values() for f in comp["features"]]
 
CATEGORY_LABELS = ["Low", "Medium", "High"]
N_QUANTILE_BINS = 3
 

 

 

In [4]:
 
# ─────────────────────────────────────────────────────────────────────────────
# Data loading
# ─────────────────────────────────────────────────────────────────────────────
 
def _left_join(left: pd.DataFrame, right: pd.DataFrame) -> pd.DataFrame:
    right_cols = [c for c in right.columns if c not in left.columns or c in JOIN_KEYS]
    return left.merge(right[right_cols], on=JOIN_KEYS, how="left").reset_index(drop=True)
 
 
def load_merged() -> pd.DataFrame:
    df_census   = preprocess_census(CENSUS_PATH)
    df_rainfall = preprocess_rainfall(RAINFALL_PATH)
    df_agri     = preprocess_agriculture(AGRI_PATH)
    df_mpi      = preprocess_mpi(MPI_PATH)
 
    df = df_census.copy()
    df = _left_join(df, df_rainfall)
    df = _left_join(df, df_agri)
    df = _left_join(df, df_mpi)
    return df

In [5]:
 
# ─────────────────────────────────────────────────────────────────────────────
# Missing-value strategy
# ─────────────────────────────────────────────────────────────────────────────
 
def impute_for_scoring(df: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    """
    Impute missing values with the state-level median before scoring.
 
    Rationale:
    - Rainfall gaps arise from boundary mismatches → neighbouring districts
      in the same state share similar climate, so state median is a sound proxy.
    - Agriculture gaps are mostly urban districts → state median of a
      predominantly rural feature remains more honest than global median.
    - MPI gaps are few (29) → state median is adequate.
 
    The original columns are preserved; imputed copies are created with the
    suffix '_imp' and used only for scoring.
    """
    df = df.copy()
    for feat in features:
        imp_col = f"{feat}_imp"
        df[imp_col] = df[feat].copy()
        # Fill with state-level median, then fall back to global median
        state_medians = df.groupby("state_key")[feat].transform("median")
        global_median = df[feat].median()
        df[imp_col] = df[imp_col].fillna(state_medians).fillna(global_median)
    return df

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# Scoring
# ─────────────────────────────────────────────────────────────────────────────
 
def normalize_features(df: pd.DataFrame, features: list[str]) -> tuple[pd.DataFrame, MinMaxScaler]:
    """
    Min-max scale the imputed feature columns to [0, 1].
    Returns the dataframe with new '_scaled' columns and the fitted scaler.
    """
    imp_cols = [f"{f}_imp" for f in features]
    scaler   = MinMaxScaler()
    scaled   = scaler.fit_transform(df[imp_cols])
 
    scaled_df = pd.DataFrame(scaled, columns=[f"{f}_scaled" for f in features], index=df.index)
    return pd.concat([df, scaled_df], axis=1), scaler
 
 
def compute_component_score(df: pd.DataFrame, component: dict) -> pd.Series:
    """
    Average the scaled features within a component, applying inversions where needed.
    Returns a [0, 1] component score series.
    """
    feature_scores = []
    inversions = component.get("invert", [False] * len(component["features"]))
 
    for feat, invert in zip(component["features"], inversions):
        col = f"{feat}_scaled"
        score = 1.0 - df[col] if invert else df[col]
        feature_scores.append(score)
 
    return pd.concat(feature_scores, axis=1).mean(axis=1)
 
 
def build_migration_pressure_score(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute the weighted migration pressure score and category labels.
 
    Formula (interpretable, deterministic):
        score = Σ weight_i × component_i
 
    where each component_i is the mean of its MinMax-scaled features.
    Components with higher weights contribute more to the final score.
    """
    df, _ = normalize_features(df, ALL_FEATURES)
 
    # Component scores
    component_scores = {}
    weighted_sum     = pd.Series(0.0, index=df.index)
 
    for name, comp in COMPONENTS.items():
        c_score = compute_component_score(df, comp)
        df[f"score_{name}"] = c_score
        component_scores[name] = c_score
        weighted_sum += comp["weight"] * c_score
 
    df["migration_pressure_score"] = weighted_sum.round(6)
 
    # Quantile-based categories (equal population in each bin)
    df["migration_pressure_category"] = pd.qcut(
        df["migration_pressure_score"],
        q=N_QUANTILE_BINS,
        labels=CATEGORY_LABELS,
    )
 
    return df
 


In [7]:

# ─────────────────────────────────────────────────────────────────────────────
# Diagnostics
# ─────────────────────────────────────────────────────────────────────────────
 
def _print_section(title: str) -> None:
    print(f"\n{'─' * 60}")
    print(f"  {title}")
    print(f"{'─' * 60}")
 
 
def report_score_distribution(df: pd.DataFrame) -> None:
    score = df["migration_pressure_score"]
 
    _print_section("Score distribution — migration_pressure_score")
    print(score.describe().round(4).to_string())
 
    _print_section("Component score means (before weighting)")
    comp_cols = [f"score_{name}" for name in COMPONENTS]
    print(df[comp_cols].describe().loc[["mean", "std", "min", "max"]].round(4).to_string())
 
    _print_section("Category distribution")
    counts  = df["migration_pressure_category"].value_counts().reindex(CATEGORY_LABELS)
    pct     = (counts / len(df) * 100).round(1)
    summary = pd.DataFrame({"count": counts, "pct": pct})
    print(summary.to_string())
 
    _print_section("Score range per category")
    for label in CATEGORY_LABELS:
        sub = df[df["migration_pressure_category"] == label]["migration_pressure_score"]
        print(f"  {label:<8}  min={sub.min():.4f}  max={sub.max():.4f}  mean={sub.mean():.4f}")
 
    _print_section("Top 10 highest-pressure districts")
    top = (
        df.nlargest(10, "migration_pressure_score")
        [["state", "district", "migration_pressure_score", "migration_pressure_category"]
         + [f"score_{n}" for n in COMPONENTS]]
    )
    print(top.to_string(index=False))
 
    _print_section("Bottom 10 lowest-pressure districts")
    bot = (
        df.nsmallest(10, "migration_pressure_score")
        [["state", "district", "migration_pressure_score", "migration_pressure_category"]
         + [f"score_{n}" for n in COMPONENTS]]
    )
    print(bot.to_string(index=False))
 
 
def plot_score_distribution(df: pd.DataFrame, out_path: str = "migration_score_distribution.png") -> None:
    """Save a two-panel diagnostic plot: histogram + category bar chart."""
    score = df["migration_pressure_score"]
    counts = df["migration_pressure_category"].value_counts().reindex(CATEGORY_LABELS)
    colors = ["#4daf4a", "#ff7f00", "#e41a1c"]   # green / amber / red
 
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle("Migration Pressure Score — Distribution", fontsize=14, fontweight="bold")
 
    # Left: histogram
    ax = axes[0]
    ax.hist(score, bins=30, edgecolor="white", color="#5577aa", linewidth=0.5)
    ax.axvline(score.mean(), color="crimson", linestyle="--", linewidth=1.2, label=f"Mean={score.mean():.3f}")
    ax.axvline(score.median(), color="orange", linestyle=":", linewidth=1.2, label=f"Median={score.median():.3f}")
    ax.set_xlabel("Migration Pressure Score")
    ax.set_ylabel("Number of districts")
    ax.set_title("Score histogram (640 districts)")
    ax.legend(fontsize=9)
 
    # Right: category bar chart
    ax = axes[1]
    bars = ax.bar(CATEGORY_LABELS, counts.values, color=colors, edgecolor="white", linewidth=0.5)
    for bar, count in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
                f"{count}\n({count / len(df) * 100:.1f}%)",
                ha="center", va="bottom", fontsize=10)
    ax.set_xlabel("Category")
    ax.set_ylabel("Number of districts")
    ax.set_title("Districts per category (quantile bins)")
    ax.set_ylim(0, counts.max() * 1.2)
 
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"\n  Plot saved → {out_path}")
    plt.close()
 

In [11]:
 
 
# ─────────────────────────────────────────────────────────────────────────────
# Entry point
# ─────────────────────────────────────────────────────────────────────────────
 
def main() -> pd.DataFrame:
    print("\nLoading and merging datasets …")
    df = load_merged()
 
    print("\nImputing missing values for scoring …")
    df = impute_for_scoring(df, ALL_FEATURES)
 
    print("Building migration pressure score …")
    df = build_migration_pressure_score(df)
 
    report_score_distribution(df)
    os.makedirs("outputs", exist_ok=True)
    plot_score_distribution(df, out_path="outputs/migration_score_distribution.png")
 
    # Final columns to keep in the output frame
    score_cols = [
        "state", "district",
        "migration_pressure_score",
        "migration_pressure_category",
        "score_climate", "score_agriculture", "score_poverty", "score_demographics",
    ]
    feature_cols = ALL_FEATURES + [f"{f}_imp" for f in ALL_FEATURES]
    output_df = df[score_cols + feature_cols].copy()
 
    print(f"\n  Final output shape: {output_df.shape}")
    output_df.to_csv("../data/processed/migration_dataset.csv", index=False)
    return output_df
 
 
if __name__ == "__main__":
    df_scored = main()
 


Loading and merging datasets …


C:\Users/asus/OneDrive/Documents/climate-migration-prediction/src\step02_preprocessing.py:212: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  result = df.groupby(["state", "district"]).apply(lambda g: pd.Series({



[MPI Columns]:
['state', 'district', 'country', 'mpi_data_source', 'population_share_of_the_district', 'multidimensional_poverty_index_mpi_of_the_country', 'multidimensional_poverty_of_the_districts_multidimensional_poverty_index_mpi__ha', 'multidimensional_poverty_of_the_districts_headcount_ratio_population_in_multidimensional_poverty_h', 'multidimensional_poverty_of_the_districts_intensity_of_deprivation_among_the_poor_a', 'multidimensional_poverty_of_the_districts_total_population_2016ᵃ', 'multidimensional_poverty_of_the_districts', 'number_of_mpi_poor_people']
[MPI] rows: 640 | cols: 7
Series([], dtype: int64)

Imputing missing values for scoring …
Building migration pressure score …

────────────────────────────────────────────────────────────
  Score distribution — migration_pressure_score
────────────────────────────────────────────────────────────
count    640.0000
mean       0.2862
std        0.0700
min        0.1119
25%        0.2351
50%        0.2802
75%        0.3344
max  